#Initialization

In [0]:

from pyspark.sql.types import *
from pyspark.sql.functions import *


#1. Read from Bronze Layer

In [0]:
df = spark.table("workspace.bronze.erp_px_cat_g1v2")

In [0]:
df.limit(10).display()

#2. Silver Transformation

##2.1 Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
      df = df.withColumn(field.name, trim(col(field.name)))

##2.2 Normalization Maintenance

In [0]:
df = df.withColumn(
    "MAINTENANCE", 
    when(upper(col("MAINTENANCE")) == "YES", lit(True))
    .when(upper(col("MAINTENANCE")) == "NO", lit(False))
    .otherwise(None)
    )

##2.3 Rename Columns

In [0]:
Renamed_Map = {
    "ID": "category_id",
    "CAT": "category",
    "SUBCAT": "subcategory",
    "MAINTENANCE" : "maintenance_flag"
}

for old_name , new_name in Renamed_Map.items():
    df = df.withColumnRenamed(old_name, new_name)


##2.4 sanity check of dataframe

In [0]:
df.limit(10).display()

#3. Write dataframe to silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_product_category")

##3.1 sanity check of silver table

In [0]:
%sql
select * from workspace.silver.erp_product_category limit 10